# 07. Auditoría de Tags Editoriales y Propuesta de Normalización

Analizamos las etiquetas reales (`post_tag`) de los articulos de Anfibia, extraídas de la base de datos original.
Comparamos las etiquetas  frente a los **tópicos semánticos latentes** (BERTopic) para diagnosticar problemas de cobertura y solapamiento.

In [1]:
# 1. Parche de compatibilidad obligatorio
import coverage.types as ct
for name, fallback in {
    'Tracer': getattr(ct, 'TTracer', object),
    'TShouldTraceFn': object,
    'TShouldStartContextFn': object,
    'TFileDisposition': object,
}.items():
    if not hasattr(ct, name):
        setattr(ct, name, fallback)

# 2. Imports
import pandas as pd
import numpy as np
import plotly.graph_objects as go
from pathlib import Path
from bertopic import BERTopic
import plotly.express as px
import plotly.graph_objects as go
import warnings
warnings.filterwarnings("ignore")

# 3. Rutas
PROJECT_ROOT = Path.cwd().resolve()
if PROJECT_ROOT.name == "notebooks":
    PROJECT_ROOT = PROJECT_ROOT.parent

DATA_PROCESSED = PROJECT_ROOT / "data" / "processed" / "corpus_anfibia_lematizado.csv"
RAW_DATA = PROJECT_ROOT / "data" / "processed" / "posteos_anfibia.csv"
MODEL_PATH = PROJECT_ROOT / "outputs" / "bertopic_hibrido" / "bertopic_hibrido_model"

OUTPUT_DIR = PROJECT_ROOT / "outputs" / "analisis_etiquetas"
OUTPUT_DIR.mkdir(exist_ok=True, parents=True)

# 4. Carga de datos
df_corpus = pd.read_csv(DATA_PROCESSED)
df_tags = pd.read_csv(RAW_DATA, usecols=["ID", "post_tag"])

# Merge para traer los tags reales
df = df_corpus.merge(df_tags, on="ID", how="left")
df["doc_id"] = df.index

# 5. Carga de modelo y predicción
topic_model = BERTopic.load(MODEL_PATH)
topics, _ = topic_model.transform(df["document_text_final"].fillna("").tolist())
df["topic"] = topics

topic_info = topic_model.get_topic_info().set_index("Topic")["Name"].to_dict()
df["topic_name"] = df["topic"].map(
    lambda t: ", ".join(topic_info.get(t, "Outlier").split("_")[1:4]) if t != -1 else "Ruido(-1)"
)

print(f"Total de notas cruzadas: {len(df):,}")

c:\Trabajos\anfibia-unsam\venv\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


Total de notas cruzadas: 3,556


## 1. Cobertura Histórica de Tags

In [2]:
total = len(df)
df["tiene_tag"] = df["post_tag"].notna() & (df["post_tag"].str.strip() != "")

con_tag = df["tiene_tag"].sum()
sin_tag = total - con_tag

print(f"Notas con etiquetas: {con_tag:,} ({(con_tag/total)*100:.1f}%)")
print(f"Notas SIN etiquetas: {sin_tag:,} ({(sin_tag/total)*100:.1f}%)")

fig = go.Figure(data=[go.Pie(labels=["Con Tags", "Sin Tags"], values=[con_tag, sin_tag], hole=.4)])
fig.update_layout(title="Cobertura de Etiquetas en el Archivo", width=500, height=400)
fig.show()

Notas con etiquetas: 1,589 (44.7%)
Notas SIN etiquetas: 1,967 (55.3%)


In [3]:


# 1. Traemos el año de publicación directo del archivo original
df_fechas = pd.read_csv(RAW_DATA, usecols=["ID", "post_year"])
df_evolucion = df.merge(df_fechas, on="ID", how="left")

# 2. Agrupamos por año: Total de notas vs Notas con tag
evolucion = (
    df_evolucion.groupby("post_year")
    .agg(
        total_notas=("doc_id", "count"),
        notas_con_tag=("tiene_tag", "sum")
    )
    .reset_index()
)
evolucion["notas_sin_tag"] = evolucion["total_notas"] - evolucion["notas_con_tag"]
evolucion["pct_con_tag"] = (evolucion["notas_con_tag"] / evolucion["total_notas"] * 100).round(1)

# Filtramos posibles años nulos o erróneos (ej. 1970)
evolucion = evolucion[evolucion["post_year"] > 2010]

# 3. Gráfico de barras apiladas
fig_evo = go.Figure()

# Barra inferior: Notas que SÍ tienen tag
fig_evo.add_trace(go.Bar(
    x=evolucion["post_year"],
    y=evolucion["notas_con_tag"],
    name="Con Etiquetas",
    marker_color='#2ca02c',
    text=evolucion["pct_con_tag"].astype(str) + "%",
    textposition='inside'
))

# Barra superior: Notas SIN tag
fig_evo.add_trace(go.Bar(
    x=evolucion["post_year"],
    y=evolucion["notas_sin_tag"],
    name="Sin Etiquetas (Vacío)",
    marker_color='#d62728'
))

fig_evo.update_layout(
    barmode='stack',
    title="<b>Notas con y sin Tags por Año</b>",
    xaxis_title="Año de Publicación",
    yaxis_title="Cantidad de Artículos",
    height=500,
    width=900,
    plot_bgcolor='rgba(0,0,0,0)',
    legend=dict(orientation="h", yanchor="bottom", y=-0.2, xanchor="center", x=0.5)
)
fig_evo.update_xaxes(type='category') # Para que los años se vean discretos
fig_evo.write_html(OUTPUT_DIR / "03_evolucion_etiquetado.html")
fig_evo.show()

## 2. Fragmentación y Uso Único de Etiquetas
Separamos las etiquetas múltiples (divididas por `;`) y analizamos su frecuencia.

In [4]:
# Desanidar tags separados por punto y coma
tags_list = (
    df["post_tag"]
    .dropna()
    .str.split(r";")
    .explode()
    .str.strip()
    .loc[lambda x: x != ""]
)

tag_counts = tags_list.value_counts()
tags_unicos = len(tag_counts)
tags_una_vez = (tag_counts == 1).sum()

print(f"Cantidad de tags únicos creados: {tags_unicos:,}")
print(f"Tags usados en 1 sola nota: {tags_una_vez:,} ({(tags_una_vez/tags_unicos)*100:.1f}%)")

print("\nTop 15 Tags más usados:")
display(tag_counts.head(15).to_frame("Frecuencia"))

Cantidad de tags únicos creados: 881
Tags usados en 1 sola nota: 444 (50.4%)

Top 15 Tags más usados:


,Frecuencia
post_tag,
Internacional,238
Javier Milei,162
Derechas,114
Zapping cultural,84
Derechos Humanos,77
UNSAM,67
Chile,65
Feminismos,62
Peronismo,61


## 3. Solapamiento: Tags vs Tópicos BERTopic
Vemos cómo un mismo tópico semántico fue etiquetado con múltiples tags manuales distintos.

In [5]:
# Armamos df desanidado completo para agrupar
df_desanidado = df.drop("post_tag", axis=1).join(
    df["post_tag"].dropna().str.split(r";").explode().str.strip().rename("tag")
).dropna(subset=["tag"])

df_desanidado = df_desanidado[df_desanidado["tag"] != ""]

# Agrupamos por tópico
solapamiento = (
    df_desanidado[df_desanidado["topic"] != -1]
    .groupby(["topic", "topic_name"])
    .agg(
        notas_con_tag=("doc_id", "nunique"),
        tags_unicos_usados=("tag", "nunique"),
        ejemplos_tags=("tag", lambda x: " | ".join(pd.Series.unique(x)[:5]))
    )
    .reset_index()
    .sort_values("notas_con_tag", ascending=False)
)

display(solapamiento.head(10))
solapamiento.to_csv(OUTPUT_DIR / "auditoria_tags_por_topico.csv", index=False)

,topic,topic_name,notas_con_tag,tags_unicos_usados,ejemplos_tags
0,0,"político, política, electoral",113,144,Escuela de Política y Gobierno | UNSAM | Brexi...
1,1,"feminismo, feminista, patriarcado",63,66,#NiUnaMenos | LGBTIQ+ | Orgullo | Memoria tran...
2,2,"fifa, futbolista, fútbol",50,54,Perfil político | Barras bravas | Boca Juniors...
4,4,"déficit, deuda, endeudamiento",43,46,Fondos buitre | Movimientos sociales | Violenc...
5,5,"pedagógico, enseñanza, escolar",38,51,Encuentro Anfibio | Feria del Libro | Día del ...
18,18,"chileno, chile, político",36,27,Diego Maradona | Chile | Estallido en Chile | ...
13,13,"papal, catolicismo, arzobispo",32,29,Papa Francisco | Rossana Reguillo | Catolicism...
10,10,"boliviano, bolivia, político",30,43,Catolicismo | Papa Francisco | Paraguay | Reli...
16,16,"brasil, brasilero, brasilia",30,16,Brasil | Dilma Rousseff | Internacional | Mund...
8,8,"epidemia, vacunar, virus",23,24,Rosariocronica | Rosario | Covid-19 | Escuela ...


Cruzemos las Notas vs la Cantidad de Tags Únicos. Y ademas calculemos un "Índice de Fragmentación" (cuántos tags se crean por cada nota) que sirva como una tercera dimensión (color): 

si la burbuja es roja, significa que los editores inventaron casi un tag nuevo por cada artículo de ese tema.

In [6]:
import plotly.express as px

MOSTRAR_POLITICA = False

# 1. Calculamos el ratio de fragmentación sobre solapamiento
solapamiento["ratio_fragmentacion"] = (solapamiento["tags_unicos_usados"] / solapamiento["notas_con_tag"]).round(2)

# 2. Filtramos tópicos con volumen suficiente y aplicamos la flag para el Tópico 0
df_ranking = solapamiento[solapamiento["notas_con_tag"] >= 15].copy()

if not MOSTRAR_POLITICA:
    df_ranking = df_ranking[df_ranking["topic"] != 0]

# 3. Ordenamos por ratio de fragmentación (ascendente para que el peor quede arriba en barras horizontales)
df_ranking = df_ranking.sort_values("ratio_fragmentacion", ascending=True).tail(15)

# 4. Bar Chart Horizontal
fig_ranking = px.bar(
    df_ranking,
    x="ratio_fragmentacion",
    y="topic_name",
    orientation="h",
    color="ratio_fragmentacion",
    color_continuous_scale="Reds",
    text="tags_unicos_usados",
    hover_data={
        "topic": True,
        "notas_con_tag": True,
        "tags_unicos_usados": True,
        "ratio_fragmentacion": False,
        "ejemplos_tags": True,
    },
    title="<b>Proporción de etiquetas por artículo</b><br><sup>El número en la barra indica la cantidad total de tags únicos inventados para ese tema.</sup>",
    labels={
        "ratio_fragmentacion": "Índice de Caos (Tags únicos / Notas)",
        "topic_name": "Palabras del Tópico",
    },
    height=650,
    template="plotly_white",
)

# Ajustes de visualización y margen
fig_ranking.update_traces(textposition="outside")
fig_ranking.update_layout(
    xaxis_range=[0, df_ranking["ratio_fragmentacion"].max() * 1.15],
    coloraxis_showscale=False,
)

# 5. Exportar y mostrar
fig_ranking.write_html(OUTPUT_DIR / "05_ranking_caos_etiquetas.html")
fig_ranking.show()

## 4. Ideas de recomendacíon


In [8]:
cuantos = 20

# 1. Buscamos el tag manual histórico más repetido (la moda) para cada tópico
tag_canonico_dict = (
    df_desanidado[df_desanidado["topic"] != -1]
    .groupby("topic")["tag"]
    .agg(lambda x: x.mode().iloc[0].title() if not x.empty else None)
    .to_dict()
)

# 2. Asignamos el tag canónico al DataFrame
df["tag_canonico_propuesto"] = df["topic"].map(tag_canonico_dict)

# 3. Fallback: Si el tópico no tenía ningún tag histórico, usamos la palabra más fuerte de BERTopic
fallback_tags = df["topic_name"].str.split(",").str[0].str.title()
df["tag_canonico_propuesto"] = df["tag_canonico_propuesto"].fillna(fallback_tags)

# Para las notas huérfanas (-1) lo dejamos en blanco
df.loc[df["topic"] == -1, "tag_canonico_propuesto"] = np.nan

print("Muestra de estandarización (De múltiples tags caóticos a un Tag Canónico):")
display(df[df["tiene_tag"] & (df["topic"] != -1)][["title_text", "post_tag", "tag_canonico_propuesto"]].sample(cuantos, random_state=42))


### 4.2 El Bypass a las Etiquetas: Recomendación Semántica (Content-Based Filtering)
# Como la auditoría demostró que las etiquetas humanas son inconsistentes, 
# el motor de recomendación de artículos relacionados se basará exclusivamente en la distancia matemática de los textos.

print("\n" + "="*60)
print("NUEVO PARADIGMA DE RECOMENDACIÓN EDITORIAL")
print("="*60)
print("""
1. MOTOR PRINCIPAL (Matriz de Similitud): 
No se usarán los tags para recomendar. En la siguiente etapa se calculará la 'Similitud Coseno' 
sobre los embeddings densos generados por SentenceTransformer. Si dos crónicas hablan de lo mismo, 
el recomendador las vinculará automáticamente, superando el problema de las notas sin etiquetar.

2. ORDENAMIENTO EN FRONT-END (UI):
Se usará la columna 'tag_canonico_propuesto' creada arriba únicamente para agrupar 
visualmente las notas en la web y darle estructura a la interfaz (ej. mostrar el badge "Feminismos" arriba del título).
""")

# 4. Exportamos el dataset enriquecido para alimentar el futuro motor de recomendación
export_cols = ["doc_id", "ID", "title_text", "post_tag", "topic", "tag_canonico_propuesto"]
df[export_cols].to_csv(OUTPUT_DIR / "corpus_etiquetas_optimizadas.csv", index=False)
print("-> Archivo 'corpus_etiquetas_optimizadas.csv' exportado.")

Muestra de estandarización (De múltiples tags caóticos a un Tag Canónico):


,title_text,post_tag,tag_canonico_propuesto
3029,Cómo afrontar la crueldad organizada,8M; Rosariocronica,Lgbtiq+
3478,Los héroes de la cruzada viril,$LIBRA; Criptogate; Javier Milei; Masculinidad...,Javier Milei
2719,Te vi en línea y te escribí,Teletrabajo; Trabajo,Trabajo
437,Con vos en el fitito azul,Día del Padre,Día Del Padre
753,La nena que jugaba en la esma,24M: Día de la memoria por la verdad y la just...,24M: Día De La Memoria Por La Verdad Y La Just...
2920,Cuándo nos robaron la primavera progresista,Elecciones 2023; Hegemonía; Javier Milei; Junt...,Javier Milei
2661,Furia roja y resistencia negra,Brasil; Internacional; Jair Bolsonaro; Lula da...,Brasil
3026,¿Quién les va a pagar a los docentes?,Docentes,Unsam
1838,¿Qué es el tokenismo cisexista?,LGBTIQ+; Memoria trans,Lgbtiq+
2749,El día más triste del puerto de Corral,Chile; Terremoto; Valdivia,Chile



NUEVO PARADIGMA DE RECOMENDACIÓN EDITORIAL

1. MOTOR PRINCIPAL (Matriz de Similitud): 
No se usarán los tags para recomendar. En la siguiente etapa se calculará la 'Similitud Coseno' 
sobre los embeddings densos generados por SentenceTransformer. Si dos crónicas hablan de lo mismo, 
el recomendador las vinculará automáticamente, superando el problema de las notas sin etiquetar.

2. ORDENAMIENTO EN FRONT-END (UI):
Se usará la columna 'tag_canonico_propuesto' creada arriba únicamente para agrupar 
visualmente las notas en la web y darle estructura a la interfaz (ej. mostrar el badge "Feminismos" arriba del título).

-> Archivo 'corpus_etiquetas_optimizadas.csv' exportado.
